<a href="https://colab.research.google.com/github/FelipeCaves/DeepLearningDuocSB/blob/main/EV01_DL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fase 1: Carga y Preprocesamiento de Datos

## Descripción y Justificación de Decisiones Técnicas

Para el desarrollo de este modelo Perceptrón Multicapa (MLP) enfocado en la clasificación de imágenes (Paltas vs. Otras Frutas), se ha diseñado un pipeline de ingesta y preprocesamiento de datos basado en las siguientes decisiones técnicas:

1. **Carga Autónoma y Reproducible:** Los datos se ingestan directamente desde un repositorio público en la nube mediante `gdown`. **Justificación:** Esto elimina la dependencia de rutas locales o permisos privados, garantizando que el entorno sea 100% reproducible por cualquier evaluador.
2. **Redimensionamiento (Resizing):** Todas las imágenes se estandarizan a una resolución de 64x64 píxeles. **Justificación:** A diferencia de las redes convolucionales (CNN), un MLP requiere aplanar la imagen en un vector 1D. Imágenes muy grandes generarían una explosión combinatoria de pesos en las capas densas, ralentizando el entrenamiento y aumentando el riesgo de *overfitting*.
3. **Partición Estricta (80% - 10% - 10%):** Dado que la función `image_dataset_from_directory` solo realiza divisiones binarias por defecto, se implementaron operaciones de tensores (`.take()` y `.skip()`) para aislar matemáticamente un 10% exclusivo para pruebas (Test Set). **Justificación:** Validar y probar con el mismo set sesga las métricas. Aislar un Test Set garantiza una evaluación sobre datos completamente invisibles para la red durante su entrenamiento.
4. **Normalización de Escala:** Se aplicó una capa `Rescaling(1./255)`. **Justificación:** Escalar los valores de los píxeles del rango [0, 255] al rango [0.0, 1.0] asegura que las variables de entrada tengan una magnitud similar, lo que estabiliza matemáticamente el descenso del gradiente y permite una convergencia más rápida del algoritmo.
5. **Optimización del Flujo de Datos (AUTOTUNE):** Se implementó paralelización asíncrona mediante `.map()` y `.prefetch(AUTOTUNE)`. **Justificación:** Esto evita cuellos de botella en el hardware. Mientras el modelo entrena un lote, el sistema ya está procesando y normalizando el siguiente en la memoria caché, maximizando la eficiencia computacional.

In [6]:
import os
import tensorflow as tf
from tensorflow.keras import layers

# 1. DESCARGA DEL ARCHIVO .ZIP PÚBLICO
id_zip = '1L66JVpBghD_9t2E9jHCHTqcM55N7M6Ho'

!gdown --id {id_zip} -O dataset.zip
!unzip -q dataset.zip -d dataset_extraido/


# 2. PARÁMETROS BASE Y RUTAS
IMG_SIZE = (64, 64)
BATCH_SIZE = 32
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

ruta_dataset = 'dataset_extraido/dataset_ev01'


# 3. LECTURA Y DIVISIÓN MATEMÁTICA (80% - 10% - 10%)
print("\n--- Iniciando partición del dataset ---")

dataset_completo = tf.keras.utils.image_dataset_from_directory(
    ruta_dataset,
    shuffle=True,
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

# Cálculo de particiones
total_lotes = len(dataset_completo)
lotes_entrenamiento = int(0.8 * total_lotes)
lotes_validacion = int(0.1 * total_lotes)

# Aislamiento de conjuntos mediante tensores
dataset_entrenamiento = dataset_completo.take(lotes_entrenamiento)
resto_dataset = dataset_completo.skip(lotes_entrenamiento)

dataset_validacion = resto_dataset.take(lotes_validacion)
dataset_prueba = resto_dataset.skip(lotes_validacion)

# 4. NORMALIZACIÓN Y OPTIMIZACIÓN DE MEMORIA (PREFETCH)

print("--- Aplicando normalización y optimización asíncrona ---")

capa_normalizacion = layers.Rescaling(1./255)

dataset_entrenamiento = dataset_entrenamiento.map(lambda x, y: (capa_normalizacion(x), y), num_parallel_calls=AUTOTUNE)
dataset_validacion = dataset_validacion.map(lambda x, y: (capa_normalizacion(x), y), num_parallel_calls=AUTOTUNE)
dataset_prueba = dataset_prueba.map(lambda x, y: (capa_normalizacion(x), y), num_parallel_calls=AUTOTUNE)

dataset_entrenamiento = dataset_entrenamiento.cache().prefetch(buffer_size=AUTOTUNE)
dataset_validacion = dataset_validacion.cache().prefetch(buffer_size=AUTOTUNE)
dataset_prueba = dataset_prueba.cache().prefetch(buffer_size=AUTOTUNE)

print("\nlsito! a entrenar la red neuranal.")

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1L66JVpBghD_9t2E9jHCHTqcM55N7M6Ho
To: /content/dataset.zip
100% 2.09M/2.09M [00:00<00:00, 197MB/s]
replace dataset_extraido/dataset_ev01/otras_frutas/img_01.jpeg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 
--- Iniciando partición del dataset ---
Found 228 files belonging to 2 classes.
--- Aplicando normalización y optimización asíncrona ---

lsito! a entrenar la red neuranal.


In [2]:
import os
print(os.listdir())

['.config', 'sample_data']
